In [10]:
# mna_final.py
"""
Final M&A Decision Model - production-ready single file.

Usage:
  - Put ALPHAVANTAGE_API_KEY and OPENAI_API_KEY in .env or .env.openAI
  - pip install python-dotenv requests pandas numpy openpyxl openai tqdm
  - python mna_final.py
  - Or import run_mna_deal and call run_mna_deal("AAPL","MSFT")
"""

import os, time, json, math, random, re
from pathlib import Path
from datetime import datetime
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from tqdm import tqdm

# Excel libraries
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, numbers
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.chart import BarChart, Reference, PieChart
from openpyxl.formatting.rule import ColorScaleRule

# Optional OpenAI client (real API)
try:
    from openai import OpenAI
except Exception:
    OpenAI = None

# ---------------- Config & Directories ----------------
RAW_DIR = Path("./data/raw"); RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = Path("./output"); OUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_SECONDS_BETWEEN_CALLS = 12.2
_last_call_time = 0.0

# ---------------- Environment loader ----------------
def load_environment():
    env_path = None
    if Path(".env").exists():
        env_path = ".env"
    elif Path(".env.openAI").exists():
        env_path = ".env.openAI"

    if env_path:
        load_dotenv(dotenv_path=env_path, override=True)
        print(f"[env] Loaded environment from {env_path}")
    else:
        print("[env] No .env/.env.openAI found. Ensure ALPHAVANTAGE_API_KEY and OPENAI_API_KEY are set.")

    av = os.getenv("ALPHAVANTAGE_API_KEY")
    oa = os.getenv("OPENAI_API_KEY")
    print("[env] ALPHAVANTAGE_API_KEY present:", bool(av))
    print("[env] OPENAI_API_KEY present:", bool(oa))
    return av, oa

ALPHAVANTAGE_API_KEY, OPENAI_API_KEY = load_environment()

# Initialize OpenAI client if key + package present
openai_client = None
if OPENAI_API_KEY and OpenAI:
    try:
        openai_client = OpenAI(api_key=OPENAI_API_KEY)
        print("[openai] Client initialized.")
    except Exception as e:
        print("[openai] Init failed:", e)
        openai_client = None
else:
    if OPENAI_API_KEY and not OpenAI:
        print("[openai] openai package not installed; install it for GPT insights.")
    openai_client = None

# ---------------- Utilities ----------------
def safe_float(x):
    try:
        return float(x)
    except Exception:
        return None

def _write_cache(path: Path, data):
    try:
        path.write_text(json.dumps(data))
    except Exception:
        pass

# ---------------- AlphaVantage request (with caching/backoff) ----------------
def av_request(function, symbol, extra_params=None, cache=True):
    global _last_call_time
    if not ALPHAVANTAGE_API_KEY:
        raise RuntimeError("ALPHAVANTAGE_API_KEY not set in environment or .env")

    fname = RAW_DIR / f"{symbol}__{function}.json"
    if cache and fname.exists():
        try:
            return json.loads(fname.read_text())
        except Exception:
            pass

    elapsed = time.time() - _last_call_time
    if elapsed < MIN_SECONDS_BETWEEN_CALLS:
        time.sleep(MIN_SECONDS_BETWEEN_CALLS - elapsed)

    base = "https://www.alphavantage.co/query"
    params = {"function": function, "symbol": symbol, "apikey": ALPHAVANTAGE_API_KEY}
    if extra_params:
        params.update(extra_params)

    backoff = 1.5
    last_exc = None
    for attempt in range(6):
        try:
            r = requests.get(base, params=params, timeout=30)
            _last_call_time = time.time()
            if r.status_code != 200:
                last_exc = RuntimeError(f"HTTP {r.status_code}")
                time.sleep(backoff); backoff *= 2; continue
            try:
                data = r.json()
            except Exception:
                data = r.text
            if isinstance(data, dict) and ("Note" in data or "Error Message" in data):
                print("[av] Rate limit or error:", data.get("Note") or data.get("Error Message"))
                time.sleep(5 * (attempt + 1))
                continue
            if cache:
                _write_cache(fname, data)
            return data
        except Exception as e:
            last_exc = e
            time.sleep(backoff)
            backoff *= 2
    raise RuntimeError(f"AlphaVantage request failed for {symbol} {function}: {last_exc}")

# ---------------- Parsing helpers ----------------
def parse_annual_reports(json_data, key='annualReports'):
    if not isinstance(json_data, dict):
        return pd.DataFrame()
    reports = json_data.get(key, []) or []
    if not reports:
        return pd.DataFrame()
    df = pd.DataFrame(reports)
    for c in df.columns:
        def conv(v):
            if isinstance(v, str) and v.replace('.', '', 1).replace('-', '', 1).isdigit():
                try: return float(v)
                except: return v
            return v
        df[c] = df[c].apply(conv)
    if 'fiscalDateEnding' in df.columns:
        df.set_index('fiscalDateEnding', inplace=True)
        df.sort_index(inplace=True)
    return df

def cagr(start, end, periods):
    if None in (start, end) or start == 0 or periods <= 0:
        return None
    try:
        return (end / start) ** (1.0 / periods) - 1.0
    except:
        return None

# ---------------- Financial metrics ----------------
def compute_financial_metrics(ticker):
    """
    Returns: dict with overview, inc_df, bal_df, cash_df, ts and computed metrics
    """
    print(f"[data] Fetching financials for {ticker} ...")
    overview = {}
    try:
        overview = av_request("OVERVIEW", ticker)
    except Exception as e:
        print(f"[warn] OVERVIEW failed for {ticker}: {e}")

    # statements
    inc_json = {}; bal_json = {}; cash_json = {}; ts_json = {}
    try: inc_json = av_request("INCOME_STATEMENT", ticker)
    except Exception: pass
    try: bal_json = av_request("BALANCE_SHEET", ticker)
    except Exception: pass
    try: cash_json = av_request("CASH_FLOW", ticker)
    except Exception: pass
    try: ts_json = av_request("TIME_SERIES_DAILY_ADJUSTED", ticker, extra_params={"outputsize":"compact"})
    except Exception: pass

    inc_df = parse_annual_reports(inc_json)
    bal_df = parse_annual_reports(bal_json)
    cash_df = parse_annual_reports(cash_json)

    metrics = {}
    metrics['ticker'] = ticker
    metrics['market_cap'] = safe_float(overview.get('MarketCapitalization'))
    metrics['shares_outstanding'] = safe_float(overview.get('SharesOutstanding'))

    # revenue & 3y CAGR
    if not inc_df.empty and 'totalRevenue' in inc_df.columns:
        revs = inc_df['totalRevenue'].dropna().astype(float)
        metrics['last_revenue'] = revs.iloc[-1]
        metrics['revenue_cagr_3y'] = cagr(revs.iloc[-4], revs.iloc[-1], 3) if len(revs) >= 4 else None
    else:
        metrics['last_revenue'] = None; metrics['revenue_cagr_3y'] = None

    def last(df, col):
        try:
            if col in df.columns and len(df[col].dropna()) > 0:
                return float(df[col].dropna().iloc[-1])
        except:
            pass
        return None

    metrics['ebitda'] = last(inc_df, 'ebitda')
    metrics['net_income'] = last(inc_df, 'netIncome')
    metrics['ebitda_margin'] = (metrics['ebitda'] / metrics['last_revenue']) if metrics['ebitda'] is not None and metrics['last_revenue'] else None
    metrics['net_margin'] = (metrics['net_income'] / metrics['last_revenue']) if metrics['net_income'] is not None and metrics['last_revenue'] else None

    metrics['total_debt'] = last(bal_df, 'totalLiabilities')
    metrics['total_equity'] = last(bal_df, 'totalShareholderEquity')
    metrics['debt_to_equity'] = (metrics['total_debt'] / metrics['total_equity']) if metrics['total_debt'] is not None and metrics['total_equity'] else None

    cur_assets = last(bal_df, 'totalCurrentAssets'); cur_liab = last(bal_df, 'totalCurrentLiabilities')
    metrics['current_ratio'] = (cur_assets / cur_liab) if cur_assets is not None and cur_liab is not None and cur_liab != 0 else None

    metrics['cash'] = last(cash_df, 'cashAndCashEquivalentsAtCarryingValue')
    op_cf = last(cash_df, 'operatingCashflow'); capex = last(cash_df, 'capitalExpenditures')
    metrics['fcf'] = (op_cf - capex) if op_cf is not None and capex is not None else None

    metrics['eps'] = (metrics['net_income'] / metrics['shares_outstanding']) if metrics['net_income'] is not None and metrics['shares_outstanding'] else None

    return {
        'overview': overview,
        'inc_df': inc_df,
        'bal_df': bal_df,
        'cash_df': cash_df,
        'ts': ts_json,
        'metrics': metrics
    }

# ---------------- Scorecard ----------------
DEFAULT_WEIGHTS = {
    'revenue_cagr_3y': 0.22,
    'ebitda_margin': 0.22,
    'fcf': 0.16,
    'debt_to_equity': 0.14,
    'current_ratio': 0.06,
    'net_margin': 0.10
}

NORMALIZATION = {
    'revenue_cagr_3y': {'min': -0.3, 'max': 0.5, 'higher': True},
    'ebitda_margin': {'min': -0.2, 'max': 0.6, 'higher': True},
    'fcf': {'min': -1e9, 'max': 1e9, 'higher': True},
    'debt_to_equity': {'min': 0.0, 'max': 4.0, 'higher': False},
    'current_ratio': {'min': 0.2, 'max': 5.0, 'higher': True},
    'net_margin': {'min': -0.5, 'max': 0.4, 'higher': True}
}

def normalize_tuned(val, metric):
    cfg = NORMALIZATION.get(metric)
    if val is None:
        return 50.0
    try:
        v = float(val)
    except:
        return 50.0
    v_clip = min(max(v, cfg['min']), cfg['max'])
    score = ((v_clip - cfg['min']) / (cfg['max'] - cfg['min'])) * 100.0
    if not cfg['higher']:
        score = 100.0 - score
    return float(score)

def compute_scorecard_tuned(metrics, weight_map=None):
    if weight_map is None:
        weight_map = DEFAULT_WEIGHTS
    per = {}
    total = 0.0
    for m, w in weight_map.items():
        val = metrics.get(m)
        s = normalize_tuned(val, m)
        per[m] = {'value': val, 'norm_score': s, 'weight': w, 'weighted': s * w}
        total += s * w
    return total, per

# ---------------- Pro-forma and financing ----------------
def build_proforma_years(acq_metrics, tgt_metrics, deal_params=None, years=5):
    if deal_params is None:
        deal_params = {}
    p = dict(deal_params)
    acq = acq_metrics['metrics']; tgt = tgt_metrics['metrics']

    # purchase_price fallback: market cap * (1 + premium)
    purchase_price = p.get('purchase_price') or ((tgt.get('market_cap') or 0.0) * (1.0 + p.get('premium_pct', 0.25)))

    # cash capacity: up to 80% of acq cash and up to 50% of purchase price
    acq_cash = acq.get('cash') or 0.0
    max_cash_use = min(acq_cash * 0.8, purchase_price * 0.5)

    # debt capacity via leverage headroom
    acq_debt = acq.get('total_debt') or 0.0
    acq_ebitda = acq.get('ebitda') or 0.0
    net_debt = acq_debt - acq_cash
    max_leverage = p.get('max_leverage', 3.5)
    max_debt_capacity = max(0.0, max_leverage * (acq_ebitda or 0.0) - net_debt)
    debt_amount = min(max_debt_capacity, max(0.0, purchase_price - max_cash_use))

    # stock residual
    stock_amount = max(0.0, purchase_price - (max_cash_use + debt_amount))

    financing = [
        {'type': 'cash', 'pct': (max_cash_use / purchase_price) if purchase_price else 0.0, 'amount': max_cash_use},
        {'type': 'debt', 'pct': (debt_amount / purchase_price) if purchase_price else 0.0, 'amount': debt_amount, 'terms': {'rate': p.get('interest_rate_on_debt', 0.06)}},
        {'type': 'stock', 'pct': (stock_amount / purchase_price) if purchase_price else 0.0, 'amount': stock_amount}
    ]

    shares_issued = 0.0
    if stock_amount > 0 and acq.get('shares_outstanding') and acq.get('market_cap'):
        acq_price = acq['market_cap'] / acq['shares_outstanding'] if acq['shares_outstanding'] else None
        if acq_price and acq_price > 0:
            shares_issued = stock_amount / acq_price

    # synergy ramps
    rev_total_pct = p.get('expected_rev_synergies_pct', 0.02)
    cost_total_pct = p.get('expected_cost_synergies_pct', 0.05)
    rev_ramp = p.get('rev_synergy_ramp') or [rev_total_pct * (i+1)/years for i in range(years)]
    cost_ramp = p.get('cost_synergy_ramp') or [cost_total_pct * (i+1)/years for i in range(years)]

    base_rev_acq = acq.get('last_revenue') or 0.0
    base_rev_tgt = tgt.get('last_revenue') or 0.0
    base_ebitda_acq = acq.get('ebitda') or 0.0
    base_ebitda_tgt = tgt.get('ebitda') or 0.0

    cumulative_shares = (acq.get('shares_outstanding') or 1.0) + shares_issued

    years_out = []
    for y in range(years):
        rev = base_rev_acq * (1 + p.get('acq_rev_growth', 0.0))**y + base_rev_tgt * (1 + p.get('tgt_rev_growth', 0.0))**y
        rev += (base_rev_acq + base_rev_tgt) * rev_ramp[y]
        ebitda = base_ebitda_acq * (1 + p.get('acq_ebitda_growth', 0.0))**y + base_ebitda_tgt * (1 + p.get('tgt_ebitda_growth', 0.0))**y
        ebitda += (base_ebitda_acq + base_ebitda_tgt) * cost_ramp[y]

        dep = 0.03 * rev
        interest = debt_amount * p.get('interest_rate_on_debt', 0.06)
        pretax = ebitda - dep - interest
        tax = p.get('tax_rate', 0.25)
        net_income = pretax * (1 - tax) if pretax is not None else None
        eps = net_income / cumulative_shares if net_income is not None and cumulative_shares else None
        net_debt_proj = (acq.get('total_debt') or 0.0) + (tgt.get('total_debt') or 0.0) + debt_amount - ((acq.get('cash') or 0.0) + (tgt.get('cash') or 0.0))
        leverage = net_debt_proj / ebitda if ebitda and ebitda > 0 else None

        years_out.append({
            'year': y + 1,
            'revenue': rev,
            'ebitda': ebitda,
            'dep': dep,
            'interest': interest,
            'pretax': pretax,
            'net_income': net_income,
            'eps': eps,
            'leverage': leverage
        })

    return {'years': years_out, 'financing': financing, 'shares_issued': shares_issued, 'purchase_price': purchase_price}

# ---------------- AI insights (OpenAI real; deterministic + seed) ----------------
def summarize_with_ai(acq, tgt):
    """
    Return {'bullets': [...], 'risk_score': int, 'recommendation': str, 'raw': str}
    Uses OpenAI if available; fallback heuristic if not (still uses metrics so unique per pair).
    """
    # Heuristic fallback
    def heuristic():
        a = acq['metrics']; t = tgt['metrics']
        bullets = []
        ai = acq['overview'].get('Industry'); ti = tgt['overview'].get('Industry')
        if ai and ti and ai == ti:
            bullets.append("Industry alignment: likely smoother integration and clearer commercial synergies.")
        elif ai and ti:
            bullets.append("Industry mismatch: expect integration and GTM complexity.")
        if (t.get('revenue_cagr_3y') or 0) > (a.get('revenue_cagr_3y') or 0):
            bullets.append("Target growing faster — potential growth boost.")
        if (t.get('ebitda_margin') or 0) > (a.get('ebitda_margin') or 0):
            bullets.append("Target has higher margins — margin uplift possible.")
        if (a.get('debt_to_equity') or 0) > 2.0:
            bullets.append("Acquirer leverage high — financing risk and covenants to check.")
        bullets.append("Key DD: customer concentration, major contracts, IP, litigation, tax.")
        score = 50
        score += 10 if (t.get('revenue_cagr_3y') or 0) > (a.get('revenue_cagr_3y') or 0) else 0
        score += 10 if (a.get('debt_to_equity') or 0) > 2.0 else 0
        score = max(0, min(100, score))
        rec = "Proceed with focused due diligence" if score <= 60 else "High risk — remediate before proceeding"
        return {'bullets': bullets, 'risk_score': score, 'recommendation': rec, 'raw': None}

    if not openai_client:
        return heuristic()

    # Build prompt with seed for slight variation
    seed = random.randint(1, 99999)
    prompt_text = (
        "You are an expert M&A analyst. Using the facts below, return strictly valid JSON with keys:\n"
        "  bullets: array of 4-6 concise actionable insights (no more than 12 words each),\n"
        "  risk_score: integer 0-100 (0 = lowest risk),\n"
        "  recommendation: one-sentence recommendation.\n\n"
        f"Seed: {seed}\n\n"
        "Acquirer:\n"
        f"  Name: {acq['overview'].get('Name','')}\n"
        f"  Industry: {acq['overview'].get('Industry','')}\n"
        f"  MarketCap: {acq['metrics'].get('market_cap')}\n"
        f"  Revenue (last): {acq['metrics'].get('last_revenue')}\n"
        f"  Revenue CAGR (3y): {acq['metrics'].get('revenue_cagr_3y')}\n"
        f"  EBITDA margin: {acq['metrics'].get('ebitda_margin')}\n"
        f"  Debt-to-Equity: {acq['metrics'].get('debt_to_equity')}\n"
        f"  Cash: {acq['metrics'].get('cash')}\n"
        f"  FCF: {acq['metrics'].get('fcf')}\n\n"
        "Target:\n"
        f"  Name: {tgt['overview'].get('Name','')}\n"
        f"  Industry: {tgt['overview'].get('Industry','')}\n"
        f"  MarketCap: {tgt['metrics'].get('market_cap')}\n"
        f"  Revenue (last): {tgt['metrics'].get('last_revenue')}\n"
        f"  Revenue CAGR (3y): {tgt['metrics'].get('revenue_cagr_3y')}\n"
        f"  EBITDA margin: {tgt['metrics'].get('ebitda_margin')}\n"
        f"  Debt-to-Equity: {tgt['metrics'].get('debt_to_equity')}\n"
        f"  Cash: {tgt['metrics'].get('cash')}\n"
        f"  FCF: {tgt['metrics'].get('fcf')}\n\n"
        "Return JSON only. Keep bullets short and focused on M&A decisioning."
    )

    try:
        resp = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt_text}],
            temperature=0.5,
            max_tokens=400
        )
        txt = resp.choices[0].message.content.strip()
        # Extract JSON substring
        m = re.search(r'(\{.*\})', txt, flags=re.DOTALL)
        js_text = m.group(1) if m else txt
        parsed = json.loads(js_text)
        bullets = parsed.get('bullets') if isinstance(parsed.get('bullets'), list) else ([parsed.get('bullets')] if parsed.get('bullets') else [])
        risk = int(parsed.get('risk_score')) if parsed.get('risk_score') is not None else None
        rec = parsed.get('recommendation') or None
        return {'bullets': bullets, 'risk_score': risk, 'recommendation': rec, 'raw': txt}
    except Exception as e:
        print("[warn] OpenAI call/parse failed:", e)
        return heuristic()

# ---------------- Excel creation (Excel-native charts) ----------------
def write_presentation_excel(acq_ticker, tgt_ticker, acq_data, tgt_data, score_acq, score_tgt, proforma, ai_result):
    ts = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
    outpath = OUT_DIR / f"{ts}_MNA_{acq_ticker}_{tgt_ticker}.xlsx"

    wb = Workbook()
    # Raw data sheet
    ws_raw = wb.active
    ws_raw.title = "Raw_Overview"
    ws_raw.append([f"Acquirer: {acq_ticker} Overview"])
    for k, v in (acq_data['overview'] or {}).items():
        ws_raw.append([k, v])
    ws_raw.append([])
    ws_raw.append([f"Target: {tgt_ticker} Overview"])
    for k, v in (tgt_data['overview'] or {}).items():
        ws_raw.append([k, v])

    # Detailed financial statements sheets for both companies
    def write_statement_sheet(prefix, inc_df, bal_df, cash_df):
        # Income
        if not inc_df.empty:
            ws_i = wb.create_sheet(f"{prefix}_Income")
            for r in dataframe_to_rows(inc_df.reset_index(), index=False, header=True):
                ws_i.append(r)
            for cell in ws_i[1]:
                cell.font = Font(bold=True)
        # Balance
        if not bal_df.empty:
            ws_b = wb.create_sheet(f"{prefix}_Balance")
            for r in dataframe_to_rows(bal_df.reset_index(), index=False, header=True):
                ws_b.append(r)
            for cell in ws_b[1]:
                cell.font = Font(bold=True)
        # Cashflow
        if not cash_df.empty:
            ws_c = wb.create_sheet(f"{prefix}_Cashflow")
            for r in dataframe_to_rows(cash_df.reset_index(), index=False, header=True):
                ws_c.append(r)
            for cell in ws_c[1]:
                cell.font = Font(bold=True)

    write_statement_sheet(acq_ticker, acq_data['inc_df'], acq_data['bal_df'], acq_data['cash_df'])
    write_statement_sheet(tgt_ticker, tgt_data['inc_df'], tgt_data['bal_df'], tgt_data['cash_df'])

    # Summary sheet
    ws_summary = wb.create_sheet("Summary")
    ws_summary.append(["Metric", acq_ticker, tgt_ticker])
    metrics_order = [
        ("Market Cap", "market_cap"),
        ("Revenue (Last)", "last_revenue"),
        ("Revenue CAGR (3y)", "revenue_cagr_3y"),
        ("EBITDA", "ebitda"),
        ("EBITDA Margin", "ebitda_margin"),
        ("Net Income", "net_income"),
        ("Net Margin", "net_margin"),
        ("Total Debt", "total_debt"),
        ("Total Equity", "total_equity"),
        ("Debt / Equity", "debt_to_equity"),
        ("Current Ratio", "current_ratio"),
        ("Cash", "cash"),
        ("FCF", "fcf"),
        ("EPS (approx)", "eps"),
        ("Composite Score", "composite_score")
    ]
    for name, key in metrics_order:
        if key == 'composite_score':
            a_val = score_acq[0]; b_val = score_tgt[0]
        else:
            a_val = acq_data['metrics'].get(key)
            b_val = tgt_data['metrics'].get(key)
        ws_summary.append([name, a_val, b_val])
    for cell in ws_summary[1]:
        cell.font = Font(bold=True)

    # Scorecards sheet
    ws_score = wb.create_sheet("Scorecards")
    ws_score.append([f"{acq_ticker} Scorecard"])
    ws_score.append(["Metric","Value","NormScore","Weight","Weighted"])
    for k, v in score_acq[1].items():
        ws_score.append([k, v['value'], v['norm_score'], v['weight'], v['weighted']])
    ws_score.append([])
    ws_score.append([f"{tgt_ticker} Scorecard"])
    ws_score.append(["Metric","Value","NormScore","Weight","Weighted"])
    for k, v in score_tgt[1].items():
        ws_score.append([k, v['value'], v['norm_score'], v['weight'], v['weighted']])

    # ProForma sheet
    ws_pro = wb.create_sheet("ProForma")
    pf_df = pd.DataFrame(proforma['years'])
    for r in dataframe_to_rows(pf_df, index=False, header=True):
        ws_pro.append(r)
    for cell in ws_pro[1]:
        cell.font = Font(bold=True)

    # Financing sheet
    ws_fin = wb.create_sheet("Financing")
    ws_fin.append(["type","pct","amount"])
    for f in proforma['financing']:
        ws_fin.append([f['type'], f.get('pct'), f.get('amount')])

    # AI_Insights sheet
    ws_ai = wb.create_sheet("AI_Insights")
    ws_ai.append(["bullet_no", "insight"])
    bullets = ai_result.get('bullets') or []
    if isinstance(bullets, list) and len(bullets) > 0:
        for i, b in enumerate(bullets):
            ws_ai.append([i+1, b])
    else:
        ws_ai.append([1, ai_result.get('raw') or ai_result.get('recommendation') or "No AI insights"])

    # Dashboard sheet
    ws_dash = wb.create_sheet("Dashboard", 0)
    ws_dash.merge_cells("A1:H1")
    ws_dash["A1"] = "M&A Decision Dashboard"
    ws_dash["A1"].font = Font(size=16, bold=True)
    ws_dash["A2"] = f"Acquirer: {acq_ticker}"
    ws_dash["C2"] = f"Target: {tgt_ticker}"
    ws_dash["A4"] = "Key Metrics (top 8)"
    # copy first 8 metrics from Summary
    for r_idx in range(2, 2 + 8):  # Summary rows 2..9
        metric = ws_summary.cell(row=r_idx, column=1).value
        a_val = ws_summary.cell(row=r_idx, column=2).value
        b_val = ws_summary.cell(row=r_idx, column=3).value
        ws_dash.append([metric, a_val, b_val])

    # AI insights to dashboard right
    ai_col = 6
    ws_dash.cell(row=3, column=ai_col, value="AI Insights").font = Font(bold=True)
    for i, b in enumerate(bullets[:6]):
        cell = ws_dash.cell(row=5 + i, column=ai_col, value=f"• {b}")
        cell.alignment = Alignment(wrap_text=True)

    # Risk & recommendation box
    risk = ai_result.get('risk_score') or 50
    rec = ai_result.get('recommendation') or "Undertake focused due diligence"
    ws_dash.cell(row=12, column=ai_col, value="Risk Score").font = Font(bold=True)
    ws_dash.cell(row=12, column=ai_col + 1, value=risk)
    ws_dash.cell(row=13, column=ai_col, value="Recommendation").font = Font(bold=True)
    rec_cell = ws_dash.cell(row=13, column=ai_col + 1, value=rec)
    rec_cell.alignment = Alignment(wrap_text=True)
    if isinstance(risk, (int, float)):
        if risk <= 35:
            rec_cell.fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
        elif risk <= 65:
            rec_cell.fill = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")
        else:
            rec_cell.fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")

    # Create Composite Score bar chart (Excel-native) on Dashboard
    try:
        # write small table to Dashboard for chart source
        base_row = 20
        ws_dash.cell(row=base_row - 1, column=1, value="Entity")
        ws_dash.cell(row=base_row - 1, column=2, value="Composite Score")
        ws_dash.cell(row=base_row, column=1, value=acq_ticker)
        ws_dash.cell(row=base_row, column=2, value=score_acq[0])
        ws_dash.cell(row=base_row + 1, column=1, value=tgt_ticker)
        ws_dash.cell(row=base_row + 1, column=2, value=score_tgt[0])

        data = Reference(ws_dash, min_col=2, min_row=base_row, max_row=base_row + 1)
        cats = Reference(ws_dash, min_col=1, min_row=base_row, max_row=base_row + 1)
        chart = BarChart()
        chart.add_data(data, titles_from_data=False)
        chart.set_categories(cats)
        chart.title = "Composite Score"
        chart.height = 7; chart.width = 12
        ws_dash.add_chart(chart, "A18")
    except Exception as e:
        print("[warn] composite chart failed:", e)

    # Create Financing pie chart
    try:
        fin_start = 30
        ws_dash.cell(row=fin_start, column=1, value="Financing Breakdown")
        for i, f in enumerate(proforma['financing']):
            ws_dash.cell(row=fin_start + i + 1, column=1, value=f['type'])
            val = f.get('pct') if f.get('pct') is not None else f.get('amount', 0.0)
            ws_dash.cell(row=fin_start + i + 1, column=2, value=val)
        labels = Reference(ws_dash, min_col=1, min_row=fin_start + 1, max_row=fin_start + len(proforma['financing']))
        data = Reference(ws_dash, min_col=2, min_row=fin_start + 1, max_row=fin_start + len(proforma['financing']))
        pie = PieChart()
        pie.add_data(data, titles_from_data=False)
        pie.set_categories(labels)
        pie.title = "Financing Breakdown"
        pie.height = 7; pie.width = 7
        ws_dash.add_chart(pie, "D18")
    except Exception as e:
        print("[warn] financing pie chart failed:", e)

    # Conditional formatting on summary numeric columns
    try:
        # apply ColorScaleRule to columns B and C rows 2..len(metrics_order)+1
        start_row = 2; end_row = 1 + len(metrics_order)
        # apply to B (acquirer) and C (target)
        rule = ColorScaleRule(start_type="percentile", start_value=10, start_color="F8696B",
                              mid_type="percentile", mid_value=50, mid_color="FFEB84",
                              end_type="percentile", end_value=90, end_color="63BE7B")
        rng_acq = f"B{start_row}:B{end_row}"
        rng_tgt = f"C{start_row}:C{end_row}"
        ws_summary.conditional_formatting.add(rng_acq, rule)
        ws_summary.conditional_formatting.add(rng_tgt, rule)
    except Exception as e:
        print("[warn] conditional formatting failed:", e)

    # Save file
    wb.save(outpath)
    print("[excel] Saved:", outpath)
    return outpath

# ---------------- Runner ----------------
def run_mna_deal(acquirer_ticker, target_ticker, deal_params=None):
    """
    Top-level function to run a single M&A analysis and generate Excel.
    """
    if deal_params is None:
        deal_params = {
            'premium_pct': 0.25,
            'interest_rate_on_debt': 0.06,
            'tax_rate': 0.25,
            'expected_rev_synergies_pct': 0.02,
            'expected_cost_synergies_pct': 0.05,
            'max_leverage': 3.5
        }

    # fetch data
    acq = compute_financial_metrics(acquirer_ticker)
    tgt = compute_financial_metrics(target_ticker)

    # scorecards
    score_acq = compute_scorecard_tuned(acq['metrics'])
    score_tgt = compute_scorecard_tuned(tgt['metrics'])

    # proforma
    proforma = build_proforma_years(acq, tgt, deal_params, years=5)

    # AI insights
    ai_res = summarize_with_ai(acq, tgt)

    # write excel
    outpath = write_presentation_excel(acquirer_ticker, target_ticker, acq, tgt, score_acq, score_tgt, proforma, ai_res)
    print("✅ Completed successfully. File:", outpath)
    return outpath

# ---------------- Run example ----------------
if __name__ == "__main__":
    # Default tickers for first run - change as needed
    acquirer = "ACN"
    target = "INFY"
    run_mna_deal(acquirer, target)


[env] Loaded environment from .env
[env] ALPHAVANTAGE_API_KEY present: True
[env] OPENAI_API_KEY present: True
[openai] Client initialized.
[data] Fetching financials for ACN ...
[data] Fetching financials for INFY ...
[excel] Saved: output\20251007T073520Z_MNA_ACN_INFY.xlsx
✅ Completed successfully. File: output\20251007T073520Z_MNA_ACN_INFY.xlsx
